# Tutorial 1_2: 2D Constrained Panel

In this tutorial, we will extend our framework to a 2D problem representing a thin panel forming part of the outer skin of a hypersonic vehicle. The panel experiences a uniform temperature increase, but its in-plane expansion is restricted because its edges are attached to a stiff frame or stiffeners. This configuration converts free thermal strain into significant thermal stress.

# Standard imports

In [ ]:
try:
    from firedrake import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *

import numpy as np
import matplotlib.pyplot as plt

# Mesh Definition

In [ ]:
Lx = 0.5
Ly = 0.1
nx = 250
ny = 50

mesh = RectangleMesh(nx, ny, Lx, Ly, quadrilateral=True)

V = FunctionSpace(mesh, 'CG', 1)
W = VectorFunctionSpace(mesh, 'CG', 1)
DG0 = FunctionSpace(mesh, 'DG', 0)

# Material Properties and Problem Definition

In 2D, we now need to account for Poisson's ratio ($\nu$). We will assume the material properties remain effectively constant over the temperature range. 

For this problem, we apply a uniform temperature change $\Delta T$ of **200°C**. We also compute the Lamé parameters $\lambda$ and $\mu$, adapting them for a 2D Plane Stress formulation. This means the expansion in the z direction is free and creates no stress. Additionally, the previously fixed ends are now only fixed in the x direction and free in the y direction, and we artificially fix in an average sense the translation in the y direction by passing the corresponding nullspace to the solver. This choice of material and problem formulation will enable us to reproduce in the 2D the result obtained in 1D.

In [ ]:
alpha = Constant(1.2e-5)
E = Constant(210e9)
nu = Constant(0.3)
T_ref = Constant(0)

mu = E / (2 * (1 + nu)) # Plane stress
lmbda_ps = E * nu / (1 - nu**2) 

T = Function(V, name='Temperature [C]')
u = Function(W, name='Displacement [m]')
von_mises_s = Function(DG0, name='von Mises Stress [Pa]')

delta_T = Constant(500)
T.assign(delta_T)

# 1: Left, 2: Right, 3: Bottom, 4: Top
# Constrain W.sub(0) (X-displacement) at the ends
# Constrain W.sub(1) (Y-displacement) at the bottom to prevent rigid body translation
bc_mech = [DirichletBC(W.sub(0), Constant(0), 1),
           DirichletBC(W.sub(0), Constant(0), 2)]

y_translation = Function(W).interpolate(as_vector([0.0, 1.0]))
nullspace = VectorSpaceBasis([y_translation])
nullspace.orthonormalize()

# Thermoelastic Solver Setup

In 2D, the derivation of the weak form used by Firedrake is more involved. We introduce here the weakly-coupled thermoelastic model, which only uses the temperature field as an input.

## Mathematical Derivation

### 1. Strong Form
In the absence of body forces and assuming quasi-static conditions, the balance of linear momentum is:
$$- \nabla \cdot \sigma = 0 \quad \text{in } \Omega$$

For a linear thermoelastic material, the stress tensor $\sigma$ is defined by the **Duhamel-Neumann constitutive law**:
$$\sigma = \lambda \text{tr}(\epsilon) \mathbf{I} + 2\mu \epsilon - (3\lambda + 2\mu)\alpha(T - T_{\text{ref}}) \mathbf{I}$$

Where:
* $\epsilon = \frac{1}{2}(\nabla \mathbf{u} + (\nabla \mathbf{u})^T)$ is the linearized strain tensor.
* $\lambda$ and $\mu$ are the **Lamé parameters**.
* $\alpha$ is the coefficient of linear thermal expansion.
* $T_{\text{ref}}$ is the reference temperature at which the material is stress-free.

### 2. Weak Form Derivation
To find the variational (weak) form, we multiply the strong form by a vector-valued test function $\mathbf{v} \in W$ and integrate over the domain $\Omega$:
$$\int_{\Omega} (- \nabla \cdot \sigma) \cdot \mathbf{v} \, dx = 0$$

Applying **integration by parts**:
$$\int_{\Omega} \sigma : \nabla \mathbf{v} \, dx - \int_{\partial \Omega} (\sigma \cdot \mathbf{n}) \cdot \mathbf{v} \, ds = 0$$

Since the stress tensor $\sigma$ is symmetric, the inner product with the gradient of the test function $\nabla \mathbf{v}$ is equivalent to the inner product with the symmetric strain operator $\epsilon(\mathbf{v})$:
$$\sigma : \nabla \mathbf{v} = \sigma : \epsilon(\mathbf{v})$$



### 3. Final Variational Statement
Assuming zero traction (Neumann) on the free boundaries and fixed (Dirichlet) conditions where specified, the boundary integral vanishes. The problem becomes: Find $\mathbf{u} \in W$ such that:
$$\int_{\Omega} \sigma(\mathbf{u}, T) : \epsilon(\mathbf{v}) \, dx = 0 \quad \forall \mathbf{v} \in W$$

In the Firedrake implementation, this is written as:
`mech_form = inner(sigma(u_mech, T), epsilon(v_mech)) * dx`

In [ ]:
u_mech = TrialFunction(W)
v_mech = TestFunction(W)

def epsilon(u):
    return 0.5 * (grad(u) + grad(u).T)

def sigma(u, T):
    thermal_strain = (2 * lmbda_ps + 2 * mu) * alpha * (T - T_ref)
    return lmbda_ps * div(u) * Identity(2) + 2*mu * epsilon(u) - thermal_strain * Identity(2)
    
mech_form = inner(sigma(u_mech, T), epsilon(v_mech)) * dx

a_mech = lhs(mech_form)
L_mech = rhs(mech_form)

mech_prob = LinearVariationalProblem(a_mech, L_mech, u, bcs=bc_mech, constant_jacobian=True)
mech_solver = LinearVariationalSolver(mech_prob, nullspace=nullspace)

# Solution and analytical validation

As expected, this 2D formulation accurately reproduce the results obtained for a fully constrained 1D bar ($\sigma = E \alpha \Delta T$).

In [ ]:
mech_solver.solve()

s = sigma(u, T)
s_xx, s_yy = s[0,0], s[1,1]
s_xy = s[0,1]
von_mises_s.project(sqrt(s_xx**2 + s_yy**2 - s_xx*s_yy + 3*s_xy**2)/1e6)

expected_sigma = float(E) * float(alpha) * float(delta_T) / 1e6
print(f"Theoretical 1D Stress: {expected_sigma:.1f} MPa")

computed_max = np.max(von_mises_s.dat.data_ro_with_halos)
print(f"Computed Max Stress: {computed_max:.1f} MPa")

plot_max = expected_sigma * 1.5 
levels = np.linspace(0, plot_max, 100)

fig, axes = plt.subplots(figsize=(10, 4))
cont = tricontourf(von_mises_s, axes=axes, levels=levels)
fig.colorbar(cont, label='von Mises Stress [MPa]')
axes.set_aspect('equal')
axes.set_title(f"Thermal Stress (Plane Stress)")

plt.show()

# Free Expansion with a Spatially Varying Temperature Field

In the previous section, the 2D panel was fully constrained in the x-direction at both ends, converting free thermal strain into stress. Now, let's explore what happens when we remove the boundary condition on the right edge, allowing the panel to expand freely.

We will apply a linear temperature gradient $T(x)$ that is 0°C at the fixed left end ($x=0$) and reaches $\Delta T$ at the free right end ($x=L_x$). 

Just like in the 1D case, because the structure is free to expand and the linear temperature field produces a compatible strain field, we expect the thermal stresses to be effectively zero.

In [ ]:
x_coord, y_coord = SpatialCoordinate(mesh)

T_linear_expr = delta_T * (x_coord / Lx)
T.interpolate(T_linear_expr)

mech_prob_free = LinearVariationalProblem(a_mech, L_mech, u, bcs=bc_mech[0], constant_jacobian=True)
mech_solver_free = LinearVariationalSolver(mech_prob_free, nullspace=nullspace)

mech_solver_free.solve()

s_free = sigma(u, T)
s_xx_free, s_yy_free = s_free[0,0], s_free[1,1]
s_xy_free = s_free[0,1]

von_mises_s.project(sqrt(s_xx_free**2 + s_yy_free**2 - s_xx_free*s_yy_free + 3*s_xy_free**2)/1e6)

computed_u_max_x = np.max(u.sub(0).dat.data_ro_with_halos)
computed_stress_max = np.max(von_mises_s.dat.data_ro_with_halos)

analytical_u_max = 0.5 * float(alpha) * float(delta_T) * Lx

print(f"Analytical Max X-Displacement:     {analytical_u_max:.5e} m")
print(f"Firedrake Max X-Displacement:      {computed_u_max_x:.5e} m")

fig, axs = plt.subplots(3, 1, figsize=(10, 12))

cont0 = tricontourf(T, axes=axs[0], cmap='inferno')
fig.colorbar(cont0, ax=axs[0], label='Temperature [C]')
axs[0].set_title("Linear Temperature Field T(x)")
axs[0].set_aspect('equal')

cont1 = tricontourf(u.sub(0), axes=axs[1], cmap='viridis')
fig.colorbar(cont1, ax=axs[1], label='X-Displacement [m]')
axs[1].set_title("Displacement u_x")
axs[1].set_aspect('equal')

cont2 = tricontourf(von_mises_s, axes=axs[2], cmap='coolwarm', levels=np.linspace(0, 2, 50))
fig.colorbar(cont2, ax=axs[2], label='von Mises Stress [MPa]')
axs[2].set_title("Thermal Stress")
axs[2].set_aspect('equal')

plt.tight_layout()
plt.show()

Apart from numerical artefacts at the corner of the boundary condition, the structure is stress-free. This because a linear temperature distribution in cartesian coordinates creates a compatible displacement field which does not require any additional elastic stress.

In [ ]:
T_non_linear_expr = delta_T * (x_coord / Lx) ** 4
T.interpolate(T_non_linear_expr)

mech_solver_free.solve()

s_free = sigma(u, T)
s_xx_free, s_yy_free = s_free[0,0], s_free[1,1]
s_xy_free = s_free[0,1]

von_mises_s.project(sqrt(s_xx_free**2 + s_yy_free**2 - s_xx_free*s_yy_free + 3*s_xy_free**2)/1e6)

computed_u_max_x = np.max(u.sub(0).dat.data_ro_with_halos)
computed_stress_max = np.max(von_mises_s.dat.data_ro_with_halos)

print(f"Firedrake Max X-Displacement:      {computed_u_max_x:.5e} m")

fig, axs = plt.subplots(3, 1, figsize=(10, 12))

cont0 = tricontourf(T, axes=axs[0], cmap='inferno')
fig.colorbar(cont0, ax=axs[0], label='Temperature [C]')
axs[0].set_title("Nonlinear Temperature Field T(x)")
axs[0].set_aspect('equal')

cont1 = tricontourf(u.sub(0), axes=axs[1], cmap='viridis')
fig.colorbar(cont1, ax=axs[1], label='X-Displacement [m]')
axs[1].set_title("Displacement u_x")
axs[1].set_aspect('equal')

cont2 = tricontourf(von_mises_s, axes=axs[2], cmap='coolwarm', levels=50)
fig.colorbar(cont2, ax=axs[2], label='von Mises Stress [MPa]')
axs[2].set_title("Thermal Stress")
axs[2].set_aspect('equal')

plt.tight_layout()
plt.show()

Here, as opposed to the 1D case previously, a nonlinear temperature distribution will create thermal stress, even if the structure is unconstrained.